In [0]:
from pyspark.sql.functions import current_timestamp, col, lit


base_path = "/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/company_house/"

tables = {
    "overview": "comp_house_overview",
    "filing_history": "comp_house_filing_history",
    "people": "comp_house_people"
}

for folder, table in tables.items():

    df = (
        spark.read
        .option("multiline", "true")
        .option("recursiveFileLookup", "true")
        .json(f"{base_path}{folder}/*")
        .withColumn("last_update_ts", current_timestamp())
        .withColumn("file_path", col("_metadata.file_path"))
    )

    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"corporate_data_lakehouse.bronze.{table}"))

In [0]:
from pyspark.sql.functions import current_timestamp, col

base_path = "/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/yfinance/"

yfinance_tables = {
    "balance_sheet": "yf_balance_sheet",
    "cashflow": "yf_cash_flow",
    "history": "yf_history",
    "income_statement": "yf_income_statement",
    "stats": "yf_stats"    
}

for folder, table in yfinance_tables.items():

    df = (
        spark.read
        .option("multiline", "true")
        .option("delta.columnMapping.mode", "name")
        .option("recursiveFileLookup", "true") # read subfiles internally
        .json(f"{base_path}{folder}/")
        .withColumn("last_update_ts", current_timestamp())
        .withColumn("file_path", col("_metadata.file_path"))
    ) 
    (df.write
        .format("delta")
        .option("delta.columnMapping.mode", "name")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"corporate_data_lakehouse.bronze.{table}"))